## 0. Импорты и загрузка базы

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from diet import (UserProfile, calculate, load_foods, search, to_fooditem,
                  FoodLog, check_portion, check_day, suggest, CONDITIONS)

pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 45)

foods = load_foods()
log = FoodLog()
print(f'База загружена: {len(foods)} продуктов')

База загружена: 30965 продуктов


## 1. Профиль человека и норма дня

Задай свои данные и болезнь. Норма считается существующим калькулятором.

In [2]:
# ↓↓↓ ЗАПОЛНИ ПОД СЕБЯ ↓↓↓
profile_data = {
    "sex": "female", "age": 45, "weight": 80,
    "height": 165, "activity": "light", "goal": "lose",
}
condition  = "diabetes_t2"   # healthy / diabetes_t2 / obesity / ckd / cvd
formula    = "who"
life_stage = "default"

profile = UserProfile(**profile_data)
target = calculate(profile, formula=formula, condition=condition, life_stage=life_stage)
print(f'{target.condition_label} · {target.goal_label}')
print(f'Цель дня: {round(target.target_kcal)} ккал | '
      f'Б {round(target.protein_g)} г | Ж {round(target.fat_g)} г | '
      f'У {round(target.carbs_g)} г | клетчатка {round(target.fiber_g)} г')


Сахарный диабет 2 типа · Снижение веса
Цель дня: 1590 ккал | Б 72 г | Ж 53 г | У 207 г | клетчатка 25 г


## 2. Поиск продукта

Введи название (можно часть, по-русски или по-английски). Таблица показывает варианты.

In [3]:
query = 'творог'   # например: chicken, рис, кефир, банан, хлеб
found = search(foods, query, limit=15)
found[['name','source','brands','kcal','protein_g','fat_g','carbs_g']]

,name,source,brands,kcal,protein_g,fat_g,carbs_g
0,Био творог клубника,off,NaN,119.000,7.6,4.2,12.7
1,Биотворог Тёма c черникой,off,NaN,113.638,7.9,4.2,10.3
2,Биотворог Тёма c черносливом,off,NaN,112.818,7.9,4.2,10.1
3,Биотворог Тёма классический,off,NaN,98.110,9.1,5.0,3.5
4,Биотворог Тёма с абрикосом,off,NaN,114.048,7.9,4.2,10.4
5,Биотворог Тёма с яблоком и морковью,off,NaN,112.408,7.9,4.2,10.0
6,Биотворог с клубникой,off,Зелёная Линия,115.000,7.6,4.2,11.2
7,Биотворог с малиной,off,вв,104.200,7.6,4.2,9.0
8,Биотворог яблоко груша,off,Сарафаново,115.000,8.1,4.5,1.5
9,Блинчики с творогом,off,Царское подворье,240.000,5.9,7.6,37.0


## 3. Добавить порцию

Укажи **индекс строки** из таблицы поиска выше (0, 1, 2...) и **граммы**. Продукт добавится в дневник, и сразу покажутся предупреждения по болезни для этой порции.

In [4]:
row_index = 0      # ← номер строки из таблицы поиска
grams     = 150    # ← сколько грамм

item = to_fooditem(found.iloc[row_index])
log.add(item, grams)
print(f'Добавлено: {item.name}, {grams} г')

# Проверки порции против болезни
res = check_portion(item, grams, condition, target_kcal=target.target_kcal)
if res:
    print('Проверки порции:')
    for r in res:
        print(' ', r)
else:
    print('Порция в пределах безопасных лимитов — без предупреждений.')

Добавлено: Био творог клубника, 150 г
Проверки порции:
  • Био творог клубника (150 г): сахар (г) = 11.2 — это 38% дневного лимита (30) при сахарный диабет 2 типа


## 4. Состояние дня

Сводка: сколько съедено vs цель, и предупреждения по накопленным за день микроэлементам.

In [5]:
print('Съедено за день:')
print(log.to_df().to_string(index=False))
print()
print('Выполнение цели:')
print(log.compare(target).to_string(index=False))
print()
day = log.totals
day_checks = check_day(day, condition, target_kcal=target.target_kcal)
if day_checks:
    print('⚠️ Предупреждения по болезни (превышение лимитов):')
    for r in day_checks:
        print(' ', r)
else:
    print('✅ Лимиты микроэлементов по болезни не превышены.')

Съедено за день:
            Продукт Источник  Граммы  ккал  Б (г)  Ж (г)  У (г) Клетчатка (г)
Био творог клубника      off     150 178.5   11.4    6.3  19.05          None

Выполнение цели:
      Нутриент  Факт  Цель  Осталось  % вып.
Калории (ккал)   178  1590      1411      11
     Белки (г)    11    72        60      16
      Жиры (г)     6    53        47      12
  Углеводы (г)    19   207       188       9
 Клетчатка (г)     0    25        25       0

✅ Лимиты микроэлементов по болезни не превышены.


## 5. Подбор под норму

Что съесть, чтобы добрать нутриент. Учитывает лимиты текущей болезни.

In [6]:
nutrient = 'protein_g'   # protein_g / fat_g / carbs_g / fiber_g / kcal
day = log.totals
goals = {'kcal': target.target_kcal, 'protein_g': target.protein_g,
         'fat_g': target.fat_g, 'carbs_g': target.carbs_g,
         'fiber_g': target.fiber_g}
need = max(goals[nutrient] - day.get(nutrient, 0), 0)
print(f'До нормы по {nutrient} осталось: {need:.0f}')
if need > 0:
    sug = suggest(nutrient, need, condition, top=5, target_kcal=target.target_kcal)
    print(sug.to_string(index=False) if len(sug) else 'Ничего не найдено.')
else:
    print('Норма уже выполнена — добирать не нужно.')


До нормы по protein_g осталось: 60
           Продукт Источник  protein_g/100г  Рекоменд. граммы  покроет (г)
  Черкизово хлысты      off            35.0               172         60.1
             Mango      off            35.0               172         60.1
        Сыр Ружетт      off            35.0               172         60.1
Соя (сухие семена)      off            34.9               172         60.1
               Соя      off            34.9               172         60.1


## 6. Управление дневником

- Отменить последнюю запись: `log.pop()`
- Очистить весь день: `log.clear()`

In [7]:
# раскомментируй при необходимости:
# log.pop()    # убрать последнюю добавленную порцию
# log.clear()  # начать день заново
print(f'Записей в дневнике: {len(log.to_df())}')

Записей в дневнике: 1


## Цикл использования

1. **Поиск** (ячейка 2): ввёл `творог` → таблица
2. **Добавить** (ячейка 3): `row_index=0, grams=150` → проверка порции
3. **Состояние** (ячейка 4): вижу выполнение и предупреждения
4. Если белка мало — **Подбор** (ячейка 5): что добрать
5. Повторяй с ячейки 2 для следующего продукта

Умные предупреждения появляются, когда порция или весь день превышают лимиты болезни (сахар/натрий/калий/фосфор/насыщ.жиры).